# Modify BAU electricity input — SISEPUEDE Morocco (capacity caps only)

## Approach

**Keep MSPs unchanged** (the df_input_1.csv calibrated baseline) and ONLY apply capacity caps to prevent NemoMod from over-investing in technologies Morocco doesn't have (nuclear, geothermal, ocean) and from inventing unrealistic hydropower expansion.

### Why this works
- The previous attempt (sum MSPs = 0.99) created LP infeasibility because coal MSP=0.63 forces 63% of total demand from coal, but coal capacity caps prevent supplying that much in years where demand grows beyond what 5 GW of coal can deliver.
- The original df_input_1.csv MSPs (coal=0.65, hydro=0.05, oil=0.03, wind=0.06, sum 0.79) leave 21% headroom for NemoMod to allocate cost-optimally. The 12:46 run with this configuration produced ~65% coal in 2018 + reasonable BAU.
- Adding tight capacity caps for unwanted techs (nuclear=0, geothermal=0, ocean=0) prevents NemoMod from over-investing.
- For gas (missing in BAU output), the cost-optimal LP picks gas naturally when coal capacity is constrained, so the mix will be coal + some gas + renewables, broadly matching IEA.

### What changes vs df_input_1.csv
- `nemomod_entc_total_annual_max_capacity_pp_<tech>_gw`: realistic ceilings (5 GW coal, 5 GW gas, 1 GW oil, 5 GW hydro, 20 GW wind, 30 GW solar, **0** nuclear/geothermal/ocean, small for biomass/biogas/waste).
- `nemomod_entc_total_annual_max_capacity_investment_pp_<tech>_gw`: realistic annual build rates (0 for coal/oil, 0.2 gas, 1 wind, 2 solar, 0.05 hydro, 0 for unwanted techs).
- **MSP NOT modified** — keeps df_input_1.csv values.


In [1]:
# Setup
import pandas as pd
import shutil
import yaml
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path('/Users/fabianfuentes/git/ssp_morocco')
INPUT_DIR   = PROJECT_DIR / 'ssp_modeling' / 'input_data'
CONFIG_PATH = PROJECT_DIR / 'ssp_modeling' / 'notebooks' / 'config_files' / 'config.yaml'

SOURCE_INPUT = INPUT_DIR / 'df_input_1.csv'   # the clean calibrated baseline
TARGET_INPUT = INPUT_DIR / 'df_input_2.csv'

print(f'Reading source: {SOURCE_INPUT.name}')
print(f'Will save to:   {TARGET_INPUT.name}')
assert SOURCE_INPUT.exists(), 'df_input_1.csv missing'

Reading source: df_input_1.csv
Will save to:   df_input_2.csv


## PARAMS

In [2]:
# Minimal caps approach — leave coal/gas/oil/wind/solar with their original (-999) caps
# Only cap the techs that NemoMod was over-investing in (hydro, biomass) or that
# Morocco doesn't have at all (nuclear, geothermal, ocean).
CAPACITY_CAP_GW = {
    'hydropower':          5.0,    # KEY: prevents 324 GW artifact
    'biogas':              0.05,
    'biomass':             0.2,
    'waste_incineration':  0.05,
    'nuclear':             0.0,    # NO nuclear plan
    'geothermal':          0.0,    # NO geothermal
    'ocean':               0.0,    # NO ocean
}

INVESTMENT_CAP_GW_PER_YEAR = {
    'hydropower':          0.05,   # ~50 MW/yr realistic
    'biogas':              0.001,
    'biomass':             0.01,
    'waste_incineration':  0.001,
    'nuclear':             0.0,
    'geothermal':          0.0,
    'ocean':               0.0,
}

UPDATE_CONFIG = True
print(f'Capacity caps (only for unwanted/over-built techs): {len(CAPACITY_CAP_GW)}')
print(f'Investment caps: {len(INVESTMENT_CAP_GW_PER_YEAR)}')
print('coal/gas/oil/wind/solar caps NOT modified — keep -999 (NemoMod cost-optimizes them).')

Capacity caps (only for unwanted/over-built techs): 7
Investment caps: 7
coal/gas/oil/wind/solar caps NOT modified — keep -999 (NemoMod cost-optimizes them).


## 1. Apply caps to df_input_1.csv → df_input_2.csv

In [3]:
df = pd.read_csv(SOURCE_INPUT)
print(f'Source shape: {df.shape}')

# Show MSP values being preserved (NOT modified)
print('\nMSP values preserved from df_input_1.csv:')
mix_cols = sorted([c for c in df.columns if c.startswith('nemomod_entc_frac_min_share_production_pp_')])
sample = df[df['time_period']==3].iloc[0]
sum_msp = 0
for c in mix_cols:
    v = sample[c]
    if v > 0:
        print(f'  {c.replace("nemomod_entc_frac_min_share_production_pp_",""):<20}: {v}')
    sum_msp += v
print(f'  SUM: {sum_msp:.2f} (NemoMod fills the rest by cost-optimization)')

# Apply capacity caps
print('\nApplying capacity ceilings (max GW at any time):')
for tech, cap in CAPACITY_CAP_GW.items():
    col = f'nemomod_entc_total_annual_max_capacity_pp_{tech}_gw'
    if col in df.columns:
        df[col] = float(cap)
        print(f'  {tech:<22}: max_cap = {cap}')

print('\nApplying investment ceilings (max GW/year new capacity):')
for tech, inv in INVESTMENT_CAP_GW_PER_YEAR.items():
    col = f'nemomod_entc_total_annual_max_capacity_investment_pp_{tech}_gw'
    if col in df.columns:
        df[col] = float(inv)
        print(f'  {tech:<22}: max_inv = {inv}')

Source shape: (56, 2442)

MSP values preserved from df_input_1.csv:
  coal                : 0.65
  hydropower          : 0.05
  oil                 : 0.03
  wind                : 0.06
  SUM: 0.79 (NemoMod fills the rest by cost-optimization)

Applying capacity ceilings (max GW at any time):
  hydropower            : max_cap = 5.0
  biogas                : max_cap = 0.05
  biomass               : max_cap = 0.2
  waste_incineration    : max_cap = 0.05
  nuclear               : max_cap = 0.0
  geothermal            : max_cap = 0.0
  ocean                 : max_cap = 0.0

Applying investment ceilings (max GW/year new capacity):
  hydropower            : max_inv = 0.05
  biogas                : max_inv = 0.001
  biomass               : max_inv = 0.01
  waste_incineration    : max_inv = 0.001
  nuclear               : max_inv = 0.0
  geothermal            : max_inv = 0.0
  ocean                 : max_inv = 0.0


## 2. Save + update config

In [4]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
if TARGET_INPUT.exists():
    backup = TARGET_INPUT.with_suffix(f'.csv.bak_{ts}')
    shutil.copy(TARGET_INPUT, backup)
    print(f'Backed up: {backup.name}')

df.to_csv(TARGET_INPUT, index=False)
print(f'Saved: {TARGET_INPUT.name}  ({TARGET_INPUT.stat().st_size:,} bytes)')

if UPDATE_CONFIG:
    cfg = yaml.safe_load(CONFIG_PATH.read_text())
    if cfg.get('ssp_input_file_name') != 'df_input_2.csv':
        cfg['ssp_input_file_name'] = 'df_input_2.csv'
        with open(CONFIG_PATH, 'w') as f:
            yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)
        print('Updated config.yaml ssp_input_file_name -> df_input_2.csv')
    else:
        print('config.yaml already pointing at df_input_2.csv')

Backed up: df_input_2.csv.bak_20260515_145157
Saved: df_input_2.csv  (1,268,203 bytes)
config.yaml already pointing at df_input_2.csv


## 3. Verification

In [5]:
df_check = pd.read_csv(TARGET_INPUT)
print(f'Verifying {TARGET_INPUT.name} shape={df_check.shape}')

# MSP preserved (not modified)
print(f'\n[A] MSPs preserved from df_input_1.csv:')
mix_cols = sorted([c for c in df_check.columns if c.startswith('nemomod_entc_frac_min_share_production_pp_')])
for c in mix_cols:
    v = df_check[df_check['time_period']==3][c].iloc[0]
    if v > 0:
        print(f'  {c.replace("nemomod_entc_frac_min_share_production_pp_",""):<20}: {v}')

# Caps
print(f'\n[B] Capacity caps:')
for tech in ['coal','gas','oil','hydropower','wind','solar','nuclear','geothermal','ocean']:
    cap_col = f'nemomod_entc_total_annual_max_capacity_pp_{tech}_gw'
    inv_col = f'nemomod_entc_total_annual_max_capacity_investment_pp_{tech}_gw'
    if cap_col in df_check.columns:
        c = df_check[df_check['time_period']==35][cap_col].iloc[0]
        i = df_check[df_check['time_period']==35][inv_col].iloc[0]
        print(f'  {tech:<22} max_cap={c:>6.2f} GW   max_inv={i:>6.3f} GW/y')

cfg = yaml.safe_load(CONFIG_PATH.read_text())
print(f'\n[C] config.ssp_input_file_name: {cfg["ssp_input_file_name"]}')

print('\nReady. Now run moroco_manager_wb_elec_demand.ipynb.')

Verifying df_input_2.csv shape=(56, 2442)

[A] MSPs preserved from df_input_1.csv:
  coal                : 0.65
  hydropower          : 0.05
  oil                 : 0.03
  wind                : 0.06

[B] Capacity caps:
  coal                   max_cap=-999.00 GW   max_inv=-999.000 GW/y
  gas                    max_cap=-999.00 GW   max_inv=-999.000 GW/y
  oil                    max_cap=-999.00 GW   max_inv=-999.000 GW/y
  hydropower             max_cap=  5.00 GW   max_inv= 0.050 GW/y
  wind                   max_cap=-999.00 GW   max_inv= 0.200 GW/y
  solar                  max_cap=-999.00 GW   max_inv=-999.000 GW/y
  nuclear                max_cap=  0.00 GW   max_inv= 0.000 GW/y
  geothermal             max_cap=  0.00 GW   max_inv= 0.000 GW/y
  ocean                  max_cap=  0.00 GW   max_inv= 0.000 GW/y

[C] config.ssp_input_file_name: df_input_2.csv

Ready. Now run moroco_manager_wb_elec_demand.ipynb.
